In [139]:
import pandas as pd
import os
from uuid import uuid4
from transformers import AutoTokenizer
from math import ceil
import requests
from bs4 import BeautifulSoup
import re
from collections import defaultdict
from tqdm import tqdm
from datetime import datetime
import random



tokenizer = AutoTokenizer.from_pretrained(
   "McGill-NLP/LLM2Vec-Meta-Llama-31-8B-Instruct-mntp" ## adjust tokenization model to the one that is used in the embedding/retriever achitecture # "McGill-NLP/LLM2Vec-Meta-Llama-31-8B-Instruct-mntp"
)

token_length = 1024 # adjust to maximal token length
document_limit = 10000
dataset = 'dev' # test

# restricting legth (make room for cls token and paragraph seperators) 
TOK_LEN = token_length - 24

In [2]:
# read  data
df = pd.read_parquet(f'/raid/deallab/SF_RAG_Data/ASQA/train.parquet')

for col in df.columns:
    print(col,':')
    print(df.loc[1, col], '\n')

ambiguous_question :
Who won the 2016 ncaa football national championship? 

qa_pairs :
[{'context': "The 13–1 Alabama Crimson Tide won the game, holding off the undefeated Clemson Tigers 45–40 in the fourth quarter. Accompanied by a talented receiving corps, Clemson's Heisman Finalist quarterback Deshaun Watson had a historic performance, setting the record for most total yards in national championship game history, with 478 yards (405 passing / 73 rushing) against the nation's third-ranked defense in Alabama, breaking the record previously set by Vince Young in the 2006 Rose Bowl. Following the game, the AP Poll also named Alabama as its top team of the season, giving Alabama their fourth title in seven seasons. Both Clemson and Alabama finished the season 14–1.", 'question': "Who won the 2016 season's ncaa football national championship?", 'short_answers': array(['Clemson Tigers', '2016 Clemson Tigers football team',
        '2016 Clemson Tigers football', 'the Tigers', 'Clemson',
 

In [58]:
divmod(60, 7)

(8, 4)

In [63]:
#helper
def get_wikipage_title(url):
    return url.split('/')[-1]

def split_list(a, max_len, o):
    o_len = len(a) + ceil(len(a)/max_len) * o 
    n = ceil(o_len/ max_len)
    k, m = divmod(o_len, n)
    return (a[i*(k-o)+min(i, m):(i+1)*k -i*o+min(i+1, m)] for i in range(n))

# api calls 
def get_document(session, title):
    url = 'https://en.wikipedia.org/w/api.php?'
    headers = {'User-Agent': 'sf_rag/1.0; lassejantsch@knu.ac.kr)'}
    params = {
        'action': 'query',
        'prop': 'revisions|extracts',
        'titles': title,
        'rvstart': '2020-02-01T00:00:00Z',
        'rvlimit': '1',
        'rvdir': 'older',
        'rvprop': 'ids|timestamp',
        'rvslots': 'main',
        'formatversion':'2',
        'format': 'json'
    }
    response = {}

    # get document text
    try:
        res =requests.get(url + '&'.join([f'{k}={v}' for k,v in params.items()]), headers= headers)
        
        #parse docuements
        res_json = res.json()
        response = [v for k,v in res_json['query']['pages'][0].items() if k in ['extract','title']]
    except:
        print(res.status_code, res.url)
    
    return response


In [120]:

class DocElement():
    def __init__(self, text, tokens, level):
        self.text = text
        self.level = int(level)
        self.tokens = tokens
        self.length = len(tokens)
        
    def __len__(self):
        return self.length
    
    def __str__(self):
        return "\t"* (self.level+1) + self.text

    def get_text(self):
        return self.text
    
    def split_doc(self, max_tokens, title):
        raise NotImplementedError

class DocSection(DocElement):
    def __init__(self, title, tokens, level):
        super().__init__(title, tokens, level)
        self.title = self.text
        self.content = []
        self.has_subsection = False
        
    
    def __str__(self):
        print_str = ''.join([str(el) for el in self.content])
        return '{0}{1}'.format("\t"*self.level + self.title,print_str)
        
        
    def update_length(self):
        len_of_content = sum([len(el) for el in self.content])
        self.length = len(self.tokens) + len_of_content
        # print(self.title,len(self.tokens), len_of_content, self.length)
    
    def get_text(self):
        text = self.title
        for element in self.content:
            text += element.get_text()
        return text
    
    def append(self, content, tokens, level, type):
        if type == 'sec':
            if level-1 == self.level:
                self.content.append(DocSection(content, tokens, level))
                self.has_subsection=True
            else:
                self.content[-1].append(content, tokens, level, type)                
        elif type=='par':
            if self.has_subsection:
                self.content[-1].append(content, tokens, level, type)
            else:
                self.content.append(DocElement(content, tokens, level))
        self.update_length()
    
    def split_doc(self, max_tokens, title=''):
        if title == '':
            title = re.search(r'#+ (.*?) #+', self.title).group(1)
        else:
            title = '{0}/{1}'.format(title, re.search(r'#+ (.*?) #+', self.title).group(1))
        title_str = 'Document: {0}\n\n'.format(title)
        title_str_len = len(tokenizer.encode(title_str, add_special_tokens=False))

        # if whole doc fits into max tokens
        if self.length + title_str_len <= max_tokens:
            return [self.get_text()]
        
        # split if not
        splitted_doc_ls = []
        current_split= title_str
        current_split_len = title_str_len
        for element in self.content:
            if len(element) == 0: continue # continue if empty element
            if current_split_len + len(element) <= max_tokens:
                #print(self.level,current_split_len, len(element), 'added to current')
                current_split += element.get_text()
                current_split_len += len(element)
            elif len(element)+title_str_len <= max_tokens:
                #print(self.level, current_split_len, len(element), 'added to new')
                splitted_doc_ls.append(current_split)
                current_split = title_str + element.get_text()
                current_split_len = title_str_len + len(element)
            else:
                #print(self.level,current_split_len, len(element), 'recursion')
                if current_split_len > title_str_len: 
                    splitted_doc_ls.append(current_split)
                    current_split = title_str
                    current_split_len = title_str_len
                splitted_doc_ls.extend(element.split_doc(max_tokens, title))
        if current_split_len > title_str_len: splitted_doc_ls.append(current_split)
        return splitted_doc_ls
                
                 

class Document(DocSection):
    def __init__(self, title, tokenizer, max_len = 1024):
        self.tokenizer = tokenizer
        self.last_level = 0
        self.max_len = max_len
        super().__init__(title, self.tokenizer.encode(title, add_special_tokens=False), 0)
    
    def add(self, content, level = None):
        if not level:
            tokens = self.tokenizer.encode(content, add_special_tokens=False)
            if len(tokens) < self.max_len - 100:
                self.append(content, tokens, self.last_level, 'par')
            else:
                token_ls = split_list(tokens, self.max_len -100, 200)
        else:
            self.append(content, self.tokenizer.encode(content, add_special_tokens=False), level, 'sec')
            self.last_level = level
        

In [136]:
# remove all html tags
def clean_html_tags(string):
    return re.sub('<[^>]*>', '', string).strip()

#pars unordered lists to text
def get_ul(ul):
    list_items = re.findall(r'<li[^>]*>(.*?)</li>', ul)
    return '\n'.join([f'* {item}' for item in list_items if item.strip()])
# pars ordered list to text
def get_ol(ol):
    list_items = re.findall(r'<li[^>]*>(.*?)</li>', ol)
    return '\n'.join([f'{id+1}. {item}' for id, item in enumerate(list_items) if item.strip()])

def get_heading(h):
    heading_type = int(re.search(r'^<h(\d)', h).group(1))
    heading_text = clean_html_tags(h.replace('\n',''))
    return f"{'#'*heading_type} {heading_text if heading_text else 'Unknown'} {'#'*heading_type}\n", heading_type -1

def parse_document(title, doc_str):
    doc_str = ' '.join(doc_str.split())
    doc_ls = re.findall(r'<p.*?</p>|<h\d.*?</h\d>|<ul.*?</ul>|<ol.*?</ol>', doc_str)
    
    parsed_doc = Document(f'# {title} #\n', tokenizer)
    for entry in doc_ls:
        if re.match(r'^<h', entry):
            if clean_html_tags(entry) in ['See also', 'References']: break # break condition when main article is over
            entry_text, level = get_heading(entry)
            parsed_doc.add(entry_text, level)
        if re.match(r'^<p', entry):
            clean_entry = clean_html_tags(entry.replace('\n',''))
            parsed_doc.add(clean_entry + '\n')
        elif re.match(r'^<ul', entry):
            clean_entry = clean_html_tags(get_ul(entry))
            parsed_doc.add(clean_entry)
        elif re.match(r'^<ol', entry):
            clean_entry = clean_html_tags(get_ol(entry))
            parsed_doc.add(clean_entry)
    
    return parsed_doc.split_doc(1024)
    

In [163]:
# create embedding document dataset.
train_embeddin_easy = pd.DataFrame(columns=['question', 'text'])
train_embedding_hard = pd.DataFrame(columns=['question', 'text_pos','text_neg'])
embedding_easy_path='/raid/deallab/SF_RAG_Data/ASQA/train/train_embeddin_easy.csv'
embedding_hard_path='/raid/deallab/SF_RAG_Data/ASQA/train/train_embedding_hard.csv'
train_embeddin_easy.to_csv(embedding_easy_path, index=False)
train_embedding_hard.to_csv(embedding_hard_path, index=False)

# create evidence / qq data for qq training
evidence_train = pd.DataFrame(columns=['text'])
evidence_train_path='/raid/deallab/SF_RAG_Data/ASQA/train/evidence_train.csv'
evidence_train.to_csv(evidence_train_path, index=False)

#creating question question pairs for retrival training (not implemented)
follow_up_train = pd.DataFrame(columns=['question', 'follow_up_questions'])
follow_up_train_path='/raid/deallab/SF_RAG_Data/ASQA/train/follow_up_train.csv'
follow_up_train.to_csv(follow_up_train_path, index=False)

# init params
session = requests.Session() # initiate session
added_evidence = set() # empty evidence set

document_limit = 10000

for idx, row in tqdm(df.iterrows(), total=min(document_limit, len(df))):
        try:
                #break when document limit is reached 
                if idx == document_limit: break
                
                #extract information from sample
                sample_id = row['sample_id']
                evidence_title_mapping = {evidence['title']:get_wikipage_title(evidence['url']) for evidence in row['wikipages']}
                base_question = row['ambiguous_question']
                follow_up_qa_mapping = {qa['question']:qa['short_answers'] for qa in row['qa_pairs']}
                follow_up_qe_mapping = {qa['question']:qa['wikipage'] for qa in row['qa_pairs'] if qa['wikipage']}
                long_answers = [answ['long_answer'] for answ in row['annotations']]
                
                # crawl documents
                evidence_docs = {}
                for title, url_title in evidence_title_mapping.items():
                        _title, text = get_document(session, url_title)
                        evidence_docs[title] = parse_document(title, text)
                        
                #fill train embedding easy
                for title, docs in evidence_docs.items():
                        for doc in docs:
                                train_embeddin_easy.loc[len(train_embeddin_easy)] = [base_question, doc]
        
                # fill train embeddin hard
                for question, title in follow_up_qe_mapping.items():
                        if title not in evidence_docs or len(evidence_docs) < 2: continue
                        rel_docs = evidence_docs[title]
                        other_docs = [doc for t, docs in evidence_docs.items() if t != title for doc in docs ]
                        for doc in rel_docs:
                                train_embedding_hard.loc[len(train_embedding_hard)] = [question, doc, random.choices(other_docs, k=1)[0]]        
                                train_embedding_hard.loc[len(train_embedding_hard)] = [question, doc, random.choices(other_docs, k=1)[0]]     
                
                # fill train evidence
                for title, docs in evidence_docs.items():
                        if title in added_evidence: continue
                        added_evidence.add(title)
                        for doc in docs:
                                evidence_train.loc[len(evidence_train)] = [doc]
                
                # fill follow up question train
                for _ in range(len(follow_up_qa_mapping)-1):
                        questions = random.sample(list(follow_up_qa_mapping.keys()), len(follow_up_qa_mapping))
                        follow_up_questions = '\n'.join([f'### {question}' for question in questions])
                        follow_up_train.loc[len(follow_up_train)] = [base_question, follow_up_questions]
        except:
                print('something went wrong')

        #save dataframes after 100 iterations
        if (idx + 1 )% 100 == 0:
                #save
                train_embeddin_easy.to_csv(embedding_easy_path, mode='a', header=False, index=False)
                train_embedding_hard.to_csv(embedding_hard_path, mode='a', header=False, index=False)
                evidence_train.to_csv(evidence_train_path, mode='a', header=False, index=False)
                follow_up_train.to_csv(follow_up_train_path, mode='a', header=False, index=False)
                
                #empty
                train_embeddin_easy = pd.DataFrame(columns=['question', 'text'])
                train_embedding_hard = pd.DataFrame(columns=['question', 'text_pos','text_neg'])
                evidence_train = pd.DataFrame(columns=['text'])
                follow_up_train = pd.DataFrame(columns=['question', 'follow_up_questions'])

        
#save
train_embeddin_easy.to_csv(embedding_easy_path, mode='a', header=False, index=False)
train_embedding_hard.to_csv(embedding_hard_path, mode='a', header=False, index=False)
evidence_train.to_csv(evidence_train_path, mode='a', header=False, index=False)
follow_up_train.to_csv(follow_up_train_path, mode='a', header=False, index=False)

  3%|▎         | 109/4353 [01:53<50:27,  1.40it/s]  

something went wrong


  5%|▍         | 197/4353 [03:43<1:43:59,  1.50s/it]

something went wrong


 11%|█         | 472/4353 [09:38<59:53,  1.08it/s]  

...

 94%|█████████▍| 4106/4353 [1:22:56<03:33,  1.15it/s]

something went wrong


 99%|█████████▊| 4295/4353 [1:26:38<01:06,  1.15s/it]

something went wrong


100%|██████████| 4353/4353 [1:27:55<00:00,  1.21s/it]


In [ ]:
# # create embedding document dataset.
# embedding_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'question', 'text'])

# # create evidence for qq training
# evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'text'])

# #creating question question pairs for retrival training (not implemented)
# qq_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions'])

# embedding_output_path='/raid/deallab/SF_RAG_Data/ASQA/embedding_train.csv'
# evidence_output_path='/raid/deallab/SF_RAG_Data/ASQA/evidence_train.csv'
# follow_up_output_path='/raid/deallab/SF_RAG_Data/ASQA/follow_up_train.csv'

# #create emptey df with heading
# embedding_df.to_csv(embedding_output_path, index=False)
# evidence_df.to_csv(evidence_output_path, index=False)
# qq_df.to_csv(follow_up_output_path, index=False)

# added_evidence = set()

# #fetched_documents = {}
# for idx, row in tqdm(df.iterrows(), total=min(document_limit, len(df))):
#     try:
#         if idx == document_limit: break # change number of document to chunk/process
#         sample_id = row['sample_id']
#         evidences = row['wikipages']
#         q1 = row['ambiguous_question']
#         q2 = defaultdict(list)
#         follow_up_question = '\n'.join([f'### {q["question"]}' for q in row['qa_pairs']])
#         for q in row['qa_pairs']:
#             question = q['question']
#             wikipage = q['wikipage']
#             if not question or not wikipage: continue
#             q2[wikipage].append(question)
#         for evidence in evidences:
#             url = evidence['url']
#             new_evidence = url not in added_evidence
#             if new_evidence:
#                 added_evidence.add(url)
                
#             page = requests.get(url)
            
#             # Create a BeautifulSoup object
#             soup = BeautifulSoup(page.text, 'html.parser')
#             # get title
#             title = soup.find(id='firstHeading').get_text()
            
#             #extract content
#             content = soup.find(class_='mw-content-ltr')
#             parsed_doc = parse_document(content)
            
#             # chunk document
#             documents = [[]]
            
#             for par in re.split(r'(?=\n#{1,4})', parsed_doc):
#                 tokenized_par = tokenizer.encode(par, add_special_tokens = False)
#                 length = len(tokenized_par)
#                 if len(documents[-1]) + length < TOK_LEN:
#                     documents[-1].extend(tokenized_par)
#                 elif length > TOK_LEN:
#                     begin = 0 
#                     while begin < length:
#                         if begin + TOK_LEN >= length:
#                             documents.append(tokenized_par[begin:])
#                             break
#                         documents.append(tokenized_par[begin:begin + TOK_LEN])
#                         begin += TOK_LEN - int(TOK_LEN * 0.1)
#                 else:
#                     documents.append(tokenized_par)
            
#             for doc in documents:
#                 doc_text = tokenizer.decode(doc)
#                 id = uuid4()
#                 #fetched_documents[url].append(id)
#                 embedding_df.loc[len(embedding_df)] = [id, sample_id, title, url, q1, doc_text]
#                 if new_evidence:
#                     evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, doc_text]
            
#             if title in q2:
#                 for question in q2[title]:
#                     for doc in documents:
#                         doc_text = tokenizer.decode(doc)
#                         id = uuid4()
#                         #fetched_documents[url].append(id)
#                         embedding_df.loc[len(embedding_df)] = [id, sample_id, title, url, question, doc_text]
#         qq_df.loc[len(qq_df)] = [uuid4(), sample_id,  q1, follow_up_question]
#     except:
#         continue
    
#     #save dataframes after 100 iterations
#     if (idx + 1 )% 100 == 0:
#         embedding_df.to_csv(embedding_output_path, mode='a', header=not os.path.exists(embedding_output_path), index=False)
#         evidence_df.to_csv(evidence_output_path, mode='a', header=not os.path.exists(evidence_output_path), index=False)
#         qq_df.to_csv(follow_up_output_path, mode='a', header=not os.path.exists(embedding_output_path), index=False)
#         embedding_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'question', 'text'])
#         evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'text'])
#         qq_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions'])
        

# embedding_df.to_csv(embedding_output_path, mode='a', header=not os.path.exists(embedding_output_path), index=False)
# evidence_df.to_csv(evidence_output_path, mode='a', header=not os.path.exists(evidence_output_path), index=False)
# qq_df.to_csv(follow_up_output_path, mode='a', header=not os.path.exists(embedding_output_path), index=False)

  0%|          | 0/4353 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (10448 > 512). Running this sequence through the model will result in indexing errors
 17%|█▋        | 728/4353 [25:28<1:52:29,  1.86s/it]

<div class="mw-heading mw-heading6"><h6 id="Tariffs">Tariffs</h6><span class="mw-editsection"><span class="mw-editsection-bracket">[</span><a href="/w/index.php?title=Economic_history_of_the_United_States&amp;action=edit&amp;section=55" title="Edit section: Tariffs"><span>edit</span></a><span class="mw-editsection-bracket">]</span></span></div>


 36%|███▌      | 1561/4353 [54:32<2:07:54,  2.75s/it]

<div class="mw-heading mw-heading6"><h6 id='Availability_of_the_§_553(b)_"interpretative"_exemption'><span id="Availability_of_the_.C2.A7_553.28b.29_.22interpretative.22_exemption"></span>Availability of the § 553(b) "interpretative" exemption</h6><span class="mw-editsection"><span class="mw-editsection-bracket">[</span><a href="/w/index.php?title=United_States_administrative_law&amp;action=edit&amp;section=11" title='Edit section: Availability of the § 553(b) "interpretative" exemption'><span>edit</span></a><span class="mw-editsection-bracket">]</span></span></div>


 47%|████▋     | 2042/4353 [1:11:22<1:01:40,  1.60s/it]

<div class="mw-heading mw-heading6"><h6 id="Smith_vs._UFC">Smith vs. UFC</h6><span class="mw-editsection"><span class="mw-editsection-bracket">[</span><a href="/w/index.php?title=Ultimate_Fighting_Championship&amp;action=edit&amp;section=19" title="Edit section: Smith vs. UFC"><span>edit</span></a><span class="mw-editsection-bracket">]</span></span></div>


 60%|█████▉    | 2603/4353 [1:30:52<46:57,  1.61s/it]  

<div class="mw-heading mw-heading6"><h6 id='The_"running_rabbits"_incident'><span id="The_.22running_rabbits.22_incident"></span>The "running rabbits" incident</h6><span class="mw-editsection"><span class="mw-editsection-bracket">[</span><a href="/w/index.php?title=Kokoda_Track_campaign&amp;action=edit&amp;section=23" title='Edit section: The "running rabbits" incident'><span>edit</span></a><span class="mw-editsection-bracket">]</span></span></div>


 67%|██████▋   | 2928/4353 [1:41:19<41:37,  1.75s/it]  

In [3]:
# #parse tables to text
# def get_table(table):
#     table_text = []
#     for i, tr in enumerate(table.find('tbody').findChildren("tr" , recursive=False)):
#         tr_text = tr.get_text()
#         tr_text = re.sub(r'\n+',';',tr_text).strip(';')
#         if not tr_text: continue
#         if table_text == []:
#             tr_text = '\n#### Table: ' + tr_text
#         table_text.append(tr_text)

#     return '\n'.join(table_text)

# # parse pars to text
# def get_p(par):
#     p_text = par.get_text()
#     p_text = p_text.replace('\n','')
#     return p_text

# def get_h(heading):
#     h = heading.find(['h1', 'h2', 'h3','h4', 'h5'])
#     try:
#         heading_type = int(re.search(r'<h(\d)', str(h)).group(1))
#     except:
#         print(heading)
#         raise
#     h_text ='\n' + ' '.join(['#'*heading_type,h.get_text()])
#     return h_text

# #pars unordered lists to text
# def get_ul(ul):
#     list_text = []
#     for li in ul.find_all('li'):
#         list_text.append('* ' + li.get_text())
#     return '\n'.join(list_text)

# # pars ordered list to text
# def get_ol(ol):
#     list_text = []
#     for i, li in enumerate(ol.find_all('li')):
#         if li.get_text():
#             list_text.append(' '.join([str(i+1),li.get_text().replace('\n','')]))
#     return '\n'.join(list_text)
        

# # parse whole document
# def parse_document(doc):
#     content = doc.find_all(['div', 'p', 'table', 'ul', 'ol'])
#     document  = []
#     for cont in content:
#         #stop condition
#         if cont.name == 'div' and cont.find(['h1', 'h2', 'h3','h4', 'h5'], id=['See_also', 'References']):
#             break
        
#         #get headining
#         if cont.name == 'div' and cont.has_attr('class') and  'mw-heading' in cont['class']:
#             document.append(get_h(cont))
#         # get par
#         elif cont.name == 'p':
#             par = get_p(cont)
#             if par:
#                 document.append(par)
#         #get ul
#         elif cont.name == 'ul':
#             document.append(get_ul(cont))
#         #get ol
#         elif cont.name == 'ol':
#             document.append(get_ol(cont))
#         #get table
#         elif cont.name == 'table':
#             if cont.has_attr('class') and 'metadata' in cont['class']: continue
#             document.append(get_table(cont))
#         # explore div
#         elif cont.name == 'div':
#             document.append(parse_document(cont))

#     return '\n'.join(document).strip('\n')



In [2]:
# read  data
df = pd.read_parquet(f'/raid/deallab/SF_RAG_Data/ASQA/dev.parquet')

for col in df.columns:
    print(col,':')
    print(df.loc[1, col], '\n')

ambiguous_question :
Who is the original artist of sound of silence? 

qa_pairs :
[{'context': 'Sounds of Silence is the second studio album by Simon & Garfunkel, released on January 17, 1966. The album\'s title is a slight modification of the title of the duo\'s first major hit, "The Sound of Silence", which originally was released as "The Sounds of Silence". The song had earlier been released in an acoustic version on the album "Wednesday Morning, 3 A.M.", and later on the soundtrack to the movie "The Graduate". Without the knowledge of Paul Simon or Art Garfunkel, electric guitars, bass and drums were overdubbed by Columbia Records staff producer Tom Wilson on June 15, 1965. This new version was released as a single in September 1965, and opens the album.', 'question': 'Who is the original artist of sound of silence, the song, released in 1964?', 'short_answers': array(['Simon & Garfunkel', 'Paul Simon and Art Garfunkel',
        'Art Garfunkel', 'Paul Simon'], dtype=object), 'wikip

In [3]:
len(df)

948

In [5]:
# create embedding document dataset.
evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'text'])
qa_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions', 'long_answers', 'short_answers'])

evidence_output_path = '/raid/deallab/SF_RAG_Data/ASQA/evidence_test.csv'
qa_output_path = '/raid/deallab/SF_RAG_Data/ASQA/qa_test.csv'
evidence_df.to_csv(evidence_output_path, index=False)
qa_df.to_csv(qa_output_path, index=False)

fetched_documents = []
for idx, row in tqdm(df.iterrows(), total=min(document_limit, len(df))):
    try: 
        if idx == document_limit: break # change number of document to chunk/process
        sample_id = row['sample_id']
        evidences = row['wikipages']
        question = row['ambiguous_question']
        follow_up_question = [q["question"] for q in row['qa_pairs']]
        long_answers = [ann['long_answer'] for ann in row['annotations']]
        short_answers = [list(ann['short_answers']) for ann in row['qa_pairs']]
        for evidence in evidences:
            url = evidence['url']
            if url in fetched_documents: continue
            fetched_documents.append(url)
            page = requests.get(url)
            
            # Create a BeautifulSoup object
            soup = BeautifulSoup(page.text, 'html.parser')
            # get title
            title = soup.find(id='firstHeading').get_text()
            
            #extract content
            content = soup.find(class_='mw-content-ltr')
            parsed_doc = parse_document(content)
            
            # chunk document
            documents = [[]]
            
            for par in re.split(r'(?=\n#{1,4})', parsed_doc):
                tokenized_par = tokenizer.encode(par, add_special_tokens = False)
                length = len(tokenized_par)
                if len(documents[-1]) + length < TOK_LEN:
                    documents[-1].extend(tokenized_par)
                elif length > TOK_LEN:
                    begin = 0 
                    while begin < length:
                        if begin + TOK_LEN >= length:
                            documents.append(tokenized_par[begin:])
                            break
                        documents.append(tokenized_par[begin:begin + TOK_LEN])
                        begin += TOK_LEN - int(TOK_LEN * 0.1)
                else:
                    documents.append(tokenized_par)
                
            # print(len(documents))
            for doc in documents:
                doc_text = tokenizer.decode(doc)
                id = uuid4()
                #fetched_documents[url].append(id)
                evidence_df.loc[len(evidence_df)] = [id, sample_id, title, url, doc_text]
            
        qa_df.loc[len(qa_df)] = [uuid4(), sample_id, question, follow_up_question, long_answers, short_answers]
    except:
        print('continueing...')
    if (idx + 1 )% 100 == 0:
        print('Saving data...')
        evidence_df.to_csv(evidence_output_path, mode='a', header=False, index=False)
        qa_df.to_csv(qa_output_path, mode='a', header=False, index=False)
        evidence_df = pd.DataFrame(columns=['id', 'sample_id','title', 'url', 'text'])
        qa_df = pd.DataFrame(columns=['id', 'sample_id', 'question', 'follow_up_questions', 'long_answers', 'short_answers'])
        

evidence_df.to_csv(evidence_output_path, mode='a', header=False,index=False)
qa_df.to_csv(qa_output_path, mode='a', header=False, index=False)              
#print(fetched_documents)
print(evidence_df.head())
print(qa_df.head())

  0%|          | 1/948 [00:04<1:11:12,  4.51s/it]

0 1


  0%|          | 2/948 [00:06<46:13,  2.93s/it]  

1 2


  0%|          | 3/948 [00:09<45:18,  2.88s/it]

...

51 2


  6%|▌         | 53/948 [01:48<22:22,  1.50s/it]

52 3


  6%|▌         | 54/948 [01:49<19:29,  1.31s/it]

53 4
